In [ ]:
# Customer Segmentation using K-Means Clustering

## Objective
Perform customer segmentation using K-Means clustering to identify distinct customer groups based on purchasing behavior. Visualize the results using PCA for dimensionality reduction.

## Dataset Columns
- age
- annual_spend
- visits_per_month
- basket_size
- days_since_last_visit
- num_categories_purchased

---

## Task 1: Data Preparation 

```python
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load the dataset
df = pd.read_csv('q2_customers.csv')

# Display basic information
print("="*60)
print("DATASET INFORMATION")
print("="*60)

print(f"\nDataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

print("\n" + "-"*40)
print("First 5 rows:")
print("-"*40)
print(df.head())

print("\n" + "-"*40)
print("Data Types:")
print("-"*40)
print(df.dtypes)

print("\n" + "-"*40)
print("Missing Value Counts:")
print("-"*40)
print(df.isnull().sum())

print("\n" + "-"*40)
print("Statistical Summary:")
print("-"*40)
print(df.describe())

# Check for any missing values
if df.isnull().sum().sum() == 0:
    print("\n✓ No missing values found in the dataset")

# Separate features for scaling (all columns are numerical)
features = df.columns.tolist()
print(f"\nFeatures to be scaled: {features}")

# Scale all features using StandardScaler
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

# Convert back to DataFrame for easier handling
df_scaled = pd.DataFrame(df_scaled, columns=features)

print("\n" + "-"*40)
print("After Scaling:")
print("-"*40)
print(f"Scaled data shape: {df_scaled.shape}")
print("\nFirst 5 rows of scaled data:")
print(df_scaled.head())

print("\n" + "-"*40)
print("Scaled Data Statistics:")
print("-"*40)
print(f"Mean of each feature (should be ~0):\n{df_scaled.mean().round(6)}")
print(f"\nStandard deviation of each feature (should be ~1):\n{df_scaled.std().round(6)}")

## Task 2: Choosing K - Elbow Method
# Compute WCSS for K = 1 to 10
wcss = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    wcss.append(kmeans.inertia_)

# Create the elbow plot
plt.figure(figsize=(12, 6))

# Plot 1: Standard elbow curve
plt.subplot(1, 2, 1)
plt.plot(K_range, wcss, 'bo-', linewidth=2, markersize=8, markerfacecolor='darkblue')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=12)
plt.title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
plt.xticks(K_range)
plt.grid(True, alpha=0.3)

# Add annotations for the elbow point
for i, (k, w) in enumerate(zip(K_range, wcss)):
    plt.annotate(f'{w:.0f}', (k, w), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

# Plot 2: Elbow curve with percentage change
plt.subplot(1, 2, 2)
wcss_diff = np.diff(wcss)
wcss_percent_change = (wcss_diff / wcss[:-1]) * 100

plt.plot(K_range[1:], wcss_percent_change, 'ro-', linewidth=2, markersize=8, markerfacecolor='darkred')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Percentage Change in WCSS (%)', fontsize=12)
plt.title('WCSS Percentage Change', fontsize=14, fontweight='bold')
plt.xticks(K_range[1:])
plt.axhline(y=10, color='green', linestyle='--', alpha=0.7, label='10% change threshold')
plt.legend()
plt.grid(True, alpha=0.3)

# Annotate the elbow point
elbow_k = 4  # Based on visual inspection
plt.annotate(f'Elbow at K={elbow_k}', (elbow_k, wcss_percent_change[elbow_k-2]), 
             textcoords="offset points", xytext=(10, -20), ha='center', fontsize=10,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))

plt.tight_layout()
plt.show()

# Print WCSS values for analysis
print("\n" + "="*60)
print("WCSS VALUES FOR DIFFERENT K")
print("="*60)
wcss_table = pd.DataFrame({
    'K': list(K_range),
    'WCSS': [f'{w:.0f}' for w in wcss],
    'WCSS_Decrease': ['-'] + [f'{wcss[i-1]-wcss[i]:.0f}' for i in range(1, len(wcss))],
    'Percent_Decrease': ['-'] + [f'{((wcss[i-1]-wcss[i])/wcss[i-1]*100):.1f}%' for i in range(1, len(wcss))]
})
print(wcss_table.to_string(index=False))

# Calculate silhouette scores for additional validation
from sklearn.metrics import silhouette_score

print("\n" + "="*60)
print("SILHOUETTE SCORES FOR VALIDATION")
print("="*60)

silhouette_scores = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(df_scaled)
    score = silhouette_score(df_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k}: Silhouette Score = {score:.4f}")

# Plot silhouette scores
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, 'go-', linewidth=2, markersize=8, markerfacecolor='darkgreen')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Score by Number of Clusters', fontsize=14, fontweight='bold')
plt.xticks(range(2, 11))
plt.grid(True, alpha=0.3)
plt.show()

## Task 3: K-Means Clustering
# Fit K-Means with chosen K (K=4)
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(df_scaled)

# Add cluster column to the original dataframe
df_with_clusters = df.copy()
df_with_clusters['Cluster'] = clusters

print("="*60)
print("K-MEANS CLUSTERING RESULTS")
print("="*60)

print(f"\nNumber of clusters: {optimal_k}")
print(f"Number of samples: {len(df_with_clusters)}")

print("\n" + "-"*40)
print("Cluster Distribution:")
print("-"*40)
cluster_counts = df_with_clusters['Cluster'].value_counts().sort_index()
for cluster in range(optimal_k):
    count = cluster_counts[cluster]
    percentage = (count / len(df_with_clusters)) * 100
    print(f"Cluster {cluster}: {count} customers ({percentage:.1f}%)")

# Display first few rows with cluster assignments
print("\n" + "-"*40)
print("First 10 rows with Cluster Assignment:")
print("-"*40)
print(df_with_clusters.head(10))

# Print cluster centroids
print("\n" + "="*60)
print("CLUSTER CENTROIDS (Scaled Space)")
print("="*60)

centroids_scaled = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=features,
    index=[f'Cluster {i}' for i in range(optimal_k)]
)
print(centroids_scaled.round(4))

# Transform centroids back to original scale for interpretation
centroids_original = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=features,
    index=[f'Cluster {i}' for i in range(optimal_k)]
)

print("\n" + "="*60)
print("CLUSTER CENTROIDS (Original Scale - for Interpretation)")
print("="*60)
print(centroids_original.round(2))

# Calculate cluster summaries
print("\n" + "="*60)
print("CLUSTER STATISTICAL SUMMARY")
print("="*60)

for cluster in range(optimal_k):
    print(f"\n{'='*40}")
    print(f"CLUSTER {cluster} - {cluster_counts[cluster]} customers")
    print(f"{'='*40}")
    
    cluster_data = df_with_clusters[df_with_clusters['Cluster'] == cluster]
    print("\nFeature Means:")
    for feature in features:
        mean_val = cluster_data[feature].mean()
        overall_mean = df[feature].mean()
        print(f"  {feature:25s}: {mean_val:10.2f} (Overall: {overall_mean:.2f})")
    
    print("\nFeature Medians:")
    for feature in features:
        median_val = cluster_data[feature].median()
        print(f"  {feature:25s}: {median_val:10.2f}")

# Visualize cluster characteristics
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Cluster Characteristics by Feature', fontsize=16, fontweight='bold')

for idx, feature in enumerate(features):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    # Boxplot for each cluster
    cluster_data = [df_with_clusters[df_with_clusters['Cluster'] == i][feature] for i in range(optimal_k)]
    bp = ax.boxplot(cluster_data, labels=[f'C{i}' for i in range(optimal_k)], patch_artist=True)
    
    # Color the boxes
    colors_box = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99']
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
    
    ax.set_title(feature, fontsize=11, fontweight='bold')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Task 4: Dimensionality Reduction with PCA
# Apply PCA to reduce data to 2 components
pca = PCA(n_components=2)
df_pca = pca.fit_transform(df_scaled)

# Create DataFrame with PCA results
df_pca_df = pd.DataFrame(df_pca, columns=['PC1', 'PC2'])

# Print explained variance ratio
print("="*60)
print("PRINCIPAL COMPONENT ANALYSIS (PCA) RESULTS")
print("="*60)

print("\n" + "-"*40)
print("Explained Variance Ratio:")
print("-"*40)
for i, ratio in enumerate(pca.explained_variance_ratio_, 1):
    print(f"PC{i}: {ratio*100:.2f}% of variance explained")
print(f"\nTotal variance explained by PC1 + PC2: {pca.explained_variance_ratio_.sum()*100:.2f}%")

# Calculate cumulative variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
print(f"\nCumulative variance: {cumulative_variance[-1]*100:.2f}%")

# Print feature loadings (components)
print("\n" + "-"*40)
print("Feature Loadings (Principal Components):")
print("-"*40)

loadings_df = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=features
)
print(loadings_df.round(4))

# Create a heatmap of loadings for better visualization
plt.figure(figsize=(10, 6))
sns.heatmap(loadings_df, annot=True, cmap='RdBu_r', center=0, 
            fmt='.3f', linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('PCA Feature Loadings Heatmap', fontsize=14, fontweight='bold')
plt.ylabel('Features')
plt.xlabel('Principal Components')
plt.tight_layout()
plt.show()

# Create a bar plot for PC1 and PC2 loadings
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PC1 Loadings
ax1 = axes[0]
colors_pc1 = ['red' if x < 0 else 'green' for x in loadings_df['PC1']]
ax1.barh(features, loadings_df['PC1'], color=colors_pc1, edgecolor='black')
ax1.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax1.set_xlabel('Loading Value', fontsize=12)
ax1.set_title('PC1 Feature Loadings', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# PC2 Loadings
ax2 = axes[1]
colors_pc2 = ['red' if x < 0 else 'green' for x in loadings_df['PC2']]
ax2.barh(features, loadings_df['PC2'], color=colors_pc2, edgecolor='black')
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel('Loading Value', fontsize=12)
ax2.set_title('PC2 Feature Loadings', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print interpretation of components
print("\n" + "="*60)
print("PCA COMPONENTS INTERPRETATION")
print("="*60)

print("\nPC1 (First Principal Component):")
positive_pc1 = loadings_df[loadings_df['PC1'] > 0]['PC1'].sort_values(ascending=False)
negative_pc1 = loadings_df[loadings_df['PC1'] < 0]['PC1'].sort_values()
print(f"\n  Strongest positive contributors: {', '.join(positive_pc1.index[:3])}")
print(f"  Strongest negative contributors: {', '.join(negative_pc1.index[:3])}")

print("\nPC2 (Second Principal Component):")
positive_pc2 = loadings_df[loadings_df['PC2'] > 0]['PC2'].sort_values(ascending=False)
negative_pc2 = loadings_df[loadings_df['PC2'] < 0]['PC2'].sort_values()
print(f"\n  Strongest positive contributors: {', '.join(positive_pc2.index[:3])}")
print(f"  Strongest negative contributors: {', '.join(negative_pc2.index[:3])}")

## Task 5: Cluster Visualization
# Add PCA components to the dataframe
df_with_clusters['PC1'] = df_pca[:, 0]
df_with_clusters['PC2'] = df_pca[:, 1]

# Create scatter plot of PC1 vs PC2 colored by cluster
plt.figure(figsize=(12, 8))

# Define colors and markers for clusters
cluster_colors = {
    0: '#FF6B6B',  # Red - High-Value Loyal
    1: '#4ECDC4',  # Teal - Bargain Shoppers
    2: '#45B7D1',  # Blue - Frequent Low-Spenders
    3: '#96CEB4'   # Green - High-Value Infrequent
}

cluster_labels = {
    0: 'Cluster 0: High-Value Loyal',
    1: 'Cluster 1: Bargain Shoppers',
    2: 'Cluster 2: Frequent Low-Spenders',
    3: 'Cluster 3: High-Value Infrequent'
}

# Scatter plot
for cluster in range(optimal_k):
    cluster_data = df_with_clusters[df_with_clusters['Cluster'] == cluster]
    plt.scatter(cluster_data['PC1'], cluster_data['PC2'], 
                c=cluster_colors[cluster], 
                label=cluster_labels[cluster],
                alpha=0.6, 
                s=50,
                edgecolors='black',
                linewidth=0.5)

# Plot cluster centroids in PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], 
            c='black', 
            marker='X', 
            s=200, 
            label='Cluster Centers',
            edgecolors='white',
            linewidth=2)

# Add labels and title
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.title('Customer Segments Visualized in 2D PCA Space', fontsize=16, fontweight='bold')
plt.legend(loc='best', fontsize=10, framealpha=0.9)
plt.grid(True, alpha=0.3)

# Add annotations for cluster interpretations
annotations = [
    (min(df_with_clusters['PC1']) + 0.5, max(df_with_clusters['PC2']) - 0.5, 
     'High Value\nInfrequent'),
    (max(df_with_clusters['PC1']) - 0.5, max(df_with_clusters['PC2']) - 0.5, 
     'High Value\nLoyal'),
    (min(df_with_clusters['PC1']) + 0.5, min(df_with_clusters['PC2']) + 0.5, 
     'Low Value\nFrequent'),
    (max(df_with_clusters['PC1']) - 0.5, min(df_with_clusters['PC2']) + 0.5, 
     'Bargain\nShoppers')
]

for x, y, text in annotations:
    plt.annotate(text, (x, y), fontsize=9, ha='center', va='center',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

plt.tight_layout()
plt.show()

# Create a second visualization: 3D-like scatter with size representing another dimension
plt.figure(figsize=(12, 8))

# Use basket_size as marker size for additional insight
sizes = df_with_clusters['basket_size'] / 100  # Scale down for visualization

for cluster in range(optimal_k):
    cluster_data = df_with_clusters[df_with_clusters['Cluster'] == cluster]
    cluster_sizes = cluster_data['basket_size'] / 100
    plt.scatter(cluster_data['PC1'], cluster_data['PC2'], 
                c=cluster_colors[cluster], 
                s=cluster_sizes,
                label=cluster_labels[cluster],
                alpha=0.5,
                edgecolors='black',
                linewidth=0.5)

plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.title('Customer Segments with Basket Size as Point Size', fontsize=16, fontweight='bold')
plt.legend(loc='best', fontsize=10, framealpha=0.9)
plt.grid(True, alpha=0.3)

# Add a note about marker size
plt.annotate('Marker size represents basket size', 
             xy=(0.02, 0.98), xycoords='axes fraction',
             fontsize=10, ha='left', va='top',
             bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))

plt.tight_layout()
plt.show()

# Summary statistics by cluster
print("\n" + "="*60)
print("CLUSTER SUMMARY STATISTICS")
print("="*60)

cluster_summary = df_with_clusters.groupby('Cluster').agg({
    'age': ['mean', 'median', 'count'],
    'annual_spend': ['mean', 'median', 'std'],
    'visits_per_month': ['mean', 'median'],
    'basket_size': ['mean', 'median'],
    'days_since_last_visit': ['mean', 'median'],
    'num_categories_purchased': ['mean', 'median']
}).round(2)

print(cluster_summary)

# Export results to CSV for business use
df_with_clusters.to_csv('customer_segments_with_clusters.csv', index=False)
print("\n✓ Results exported to 'customer_segments_with_clusters.csv'")

# Create a business recommendation table
print("\n" + "="*60)
print("BUSINESS RECOMMENDATIONS BY CLUSTER")
print("="*60)

recommendations = pd.DataFrame({
    'Cluster': ['0', '1', '2', '3'],
    'Segment Name': ['High-Value Loyal', 'Bargain Shoppers', 'Frequent Low-Spenders', 'High-Value Infrequent'],
    'Size (%)': [f"{cluster_counts[i]/len(df)*100:.1f}%" for i in range(4)],
    'Primary Strategy': [
        'Retention & Cross-sell',
        'Increase frequency & Bulk offers',
        'Upsell & Increase basket size',
        'Win-back & Premium service'
    ],
    'Recommended Action': [
        'VIP program, personalized recommendations, early access to sales',
        'Bundle discounts, loyalty points, subscription model',
        'Gamification, "frequently bought together" suggestions, spend thresholds',
        'Dedicated account manager, seasonal campaigns, exclusive events'
    ]
})
print(recommendations.to_string(index=False))